# Fringes and band bending across the (+⁴−) | (+⁵) wall: a hybrid of the two models

Two calculations of this repository describe the stacking wall of six-layer graphite from opposite sides:

* `stacking_wall.py`: the k·p SWMcC Hamiltonian interpolated across the wall, solved in plane waves. It has the wall as a lateral object, hence the quasiparticle-interference fringes on the (+⁴−) side, but no doping and no tip.
* `stacking_wall_tibb.py`: the Poisson solver of the [bandbending](https://github.com/nemeslab-MFA/bandbending) engine (Hartree + γ0/γ1 tight binding) in the local approximation: laterally uniform stacks at several (+⁴−) weights *s* under the same tip, with the doping fitted on the (+⁵) side. It has the electrostatics but no wall, hence no fringes.

This notebook **reads the Poisson solver outputs** (`examples/stacking_wall_tibb_TAG*.json`: the self-consistent layer potentials under the apex and in the far field, the Fermi level and the simulated spectra of every stack) and feeds the far-field layer potentials, as position-dependent on-site energies along the measured line, into the k·p wall model. The result is a top-layer LDOS(E, x) that carries the fringes of the wall together with the band shift of the stacks. The tip-induced part of the potential moves with the tip and is treated separately at the end.

Energies: the engine works in the frame of each stack (its charge-neutrality point at 0, Fermi level μ0); here everything is referred to the common Fermi level, so the on-site energy of layer *l* is V_l = U_far,l − μ0.

In [ ]:
import glob, json, os, sys, time
import numpy as np
import matplotlib.pyplot as plt

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'examples' else os.getcwd()
sys.path.insert(0, ROOT)
EXAMPLES = os.path.join(ROOT, 'examples')
import stacking_wall as sw

TAGS = ['514', '940']
# RHK line spectra for the measured panels (optional; the json files hold only three measured spectra per line)
SM4 = {
    '514': os.environ.get('SM4_514', ''),
    '940': os.environ.get('SM4_940', ''),
}
plt.rcParams.update({'font.size': 9})
INK, MUTED, BLUE, ORANGE, GREY = '#1d1c1a', '#605d57', '#1f6fb2', '#d95926', '#52514e'

## 1. The Poisson solver outputs

For every stack (s = 0 is the (+⁵) stack of the contact-potential scan at the fitted value, s = 1 the (+⁴−) stack, the others the wall stacks) the engine stored, per bias point, the layer potential energies under the apex `U0` and in the far field `U_far`, the Fermi level `mu0` of the stack, the fixed charge per layer `doping`, and the simulated top-layer spectrum.

In [ ]:
def load_series(tag):
    """s -> record of the Poisson solver for that stack, plus the settings and the measured spectra."""
    main = json.load(open(os.path.join(EXAMPLES, f'stacking_wall_tibb_{tag}.json')))
    best = main['scan'][f"{main['best_cpot']:+.3f}"]
    recs = {0.0: best, 1.0: main['runs']['sf']}
    for f in [os.path.join(EXAMPLES, f'stacking_wall_tibb_{tag}.json')] + sorted(glob.glob(os.path.join(EXAMPLES, f'stacking_wall_tibb_{tag}_s*.json'))):
        for k, r in json.load(open(f)).get('runs', {}).items():
            if k.startswith('wall_s') and r.get('s') is not None:
                recs[float(r['s'])] = r
    out = {}
    for s, r in sorted(recs.items()):
        sweep = r['sweep']
        out[s] = dict(bias=np.array(sweep['bias']), U0=np.array(sweep['U0']), U_far=np.array(sweep['U_far']).mean(axis=0),
                      mu0=sweep['mu0'], doping=np.array(sweep['doping']), didv=np.array(r['didv']),
                      didv_frozen=np.array(r['didv_frozen']), scale=r['scale'], residual=r['residual'])
    fit = json.load(open(os.path.join(EXAMPLES, f'stacking_wall_{tag}.json')))['fit']
    return out, main['settings'], {k: np.array(v) for k, v in main['measured'].items()}, np.array(main['fine_bias']), fit, main['best_cpot']

series = {t: load_series(t) for t in TAGS}
for t in TAGS:
    rec, settings, meas, fine, fit, cpot = series[t]
    print(f'line spectrum {t}: contact potential {cpot:+.2f} eV, fermi_level {1e3*settings["fermi_level"]:+.1f} meV, '
          f'wall x0 = {fit["x0"]:.1f} nm, l = {fit["ell"]:.1f} nm; stacks at s = {sorted(rec)}')
    print('    s     mu0     V_l = U_far,l - mu0 (meV), layer 1 (top) .. 6        residual')
    for s, r in rec.items():
        V = (r['U_far'] - r['mu0']) * 1e3
        print(f'  {s:4.2f}  {1e3*r["mu0"]:+6.1f}   ' + ' '.join(f'{v:+6.1f}' for v in V) + f'      {r["residual"]:.3f}')

In [ ]:
fig, axs = plt.subplots(len(TAGS), 3, figsize=(14, 4.2 * len(TAGS)), squeeze=False)
for row, t in enumerate(TAGS):
    rec, settings, meas, fine, fit, cpot = series[t]
    s_vals = np.array(sorted(rec))
    V = np.array([rec[s]['U_far'] - rec[s]['mu0'] for s in s_vals]) * 1e3       # (n_s, 6)
    ax = axs[row, 0]
    for l in range(6):
        ax.plot(s_vals, V[:, l], 'o-', ms=3, label=f'layer {l + 1}' + (' (top)' if l == 0 else ''))
    ax.set(xlabel='s, (+4−) weight of the stack', ylabel='V_l = U_far,l − μ0 (meV)', title=f'{t}: far-field layer potentials of the stacks')
    ax.legend(fontsize=7, frameon=False, ncol=2)
    ax = axs[row, 1]
    for s in s_vals:
        r = rec[s]
        ax.plot(r['bias'], 1e3 * (r['U0'][:, 0] - r['U_far'][0]), '-', lw=1.2, label=f's = {s:.2f}')
    ax.axhline(0, color=MUTED, lw=0.5)
    ax.set(xlabel='sample bias (V)', ylabel='U_1(apex) − U_1(far) (meV)', title='tip-induced bending of the top layer')
    ax.legend(fontsize=7, frameon=False, ncol=2)
    ax = axs[row, 2]
    step = 1.1 * max(rec[s]['didv'].max() * rec[0.0]['scale'] for s in s_vals) * 1e3
    for i, s in enumerate(s_vals):
        ax.plot(fine, rec[s]['didv'] * rec[0.0]['scale'] * 1e3 + i * step, lw=1.1, label=f's = {s:.2f}')
    ax.set(xlabel='sample bias (V)', ylabel='simulated dI/dV, offset', title='simulated top-layer spectra under the apex')
    ax.legend(fontsize=7, frameon=False, ncol=2)
fig.tight_layout()

## 2. Layer potentials along the measured line

The wall profile fitted to the measured line, s(x) = [1 − tanh((x − x0)/l)]/2, maps the stacks onto positions; V_l(x) is interpolated linearly in s between the computed stacks. The k·p model lives on a periodic cell of length L with the wall at x = 0 and a return wall at ±L/2 (`stacking_wall.wall_profile`), so the same construction is applied to that periodic profile.

In [ ]:
def potential_of_s(rec):
    s_vals = np.array(sorted(rec))
    V = np.array([rec[s]['U_far'] - rec[s]['mu0'] for s in s_vals])                 # (n_s, 6) eV
    def f(s):
        s = np.asarray(s, dtype=float)
        return np.array([np.interp(s, s_vals, V[:, l]) for l in range(6)])         # (6, ...)
    return f

fig, axs = plt.subplots(1, len(TAGS), figsize=(6 * len(TAGS), 3.6), squeeze=False)
for col, t in enumerate(TAGS):
    rec, settings, meas, fine, fit, cpot = series[t]
    x = np.linspace(0, 2 * fit['x0'] + 20, 400)
    s_x = 0.5 * (1 - np.tanh((x - fit['x0']) / fit['ell']))
    Vx = potential_of_s(rec)(s_x) * 1e3
    ax = axs[0, col]
    for l in range(6):
        ax.plot(x, Vx[l], lw=1.2, label=f'layer {l + 1}')
    ax.axvline(fit['x0'], color=MUTED, ls='--', lw=0.8)
    ax.set(xlabel='distance along the line (nm)', ylabel='V_l(x) (meV)', title=f'{t}: on-site energies along the line, E_F = 0')
    ax.legend(fontsize=7, frameon=False, ncol=3)
fig.tight_layout()

## 3. Hybrid: the k·p wall model with the Poisson potentials

`stacking_wall.ldos_wall` with `layer_potential_profile`: the periodic cell is 480 nm long, the wall length is the fitted *l*, and the potentials follow the periodic profile. The k_y mesh is a little coarser than in the paper-style figures (0.0015 Å⁻¹) to keep the run to a few minutes per line on four cores; the Gaussian broadening is 5 meV. Results are cached in `examples/stacking_wall_hybrid_TAG.npz`.

In [ ]:
HYBRID = {}
for t in TAGS:
    rec, settings, meas, fine, fit, cpot = series[t]
    cache = os.path.join(EXAMPLES, f'stacking_wall_hybrid_{t}.npz')
    if os.path.exists(cache):
        HYBRID[t] = sw.load_ldos(cache)
        print(f'[{t}] loaded {cache}')
        continue
    L, ell = 4800.0, 10.0 * fit['ell']
    V_of_s = potential_of_s(rec)
    profile = lambda x, ell=ell: sw.wall_profile(x, ell, L)
    layer_potential_profile = lambda x, V_of_s=V_of_s, profile=profile: V_of_s(profile(x))
    t0 = time.time()
    res = sw.ldos_wall(wall_length=ell, length=L, p_cut=0.15, ky_max=0.15, dky=0.0015, sigma=0.005,
                       energy=np.arange(-0.5, 0.5 + 1e-9, 0.0025), x_out=np.arange(-1000.0, 1000.0 + 1e-9, 5.0),
                       layer_potential_profile=layer_potential_profile)
    print(f'[{t}] hybrid LDOS in {time.time() - t0:.0f} s')
    sw.save_ldos(cache, res)
    HYBRID[t] = res

## 4. Experiment, local approximation and hybrid side by side

The local-approximation map (`examples/stacking_wall_tibb_map_TAG.npz`, from `stacking_wall_tibb_map.py`) is the simulated dI/dV of the uniform stacks placed along the same profile. The hybrid LDOS is drawn with its energy axis shifted rigidly so that its (+⁵) flat band far from the wall sits at the measured energy (the SWMcC flat band is a few meV above the neutrality point, the γ0/γ1 one at it); the measured line is shown when the `.sm4` file is available.

In [ ]:
def peak_track(x, E, Z, E0, half=0.09, h_min=0.15):
    """energy of the maximum within E0 +- half along x, NaN where the peak has faded below h_min of its far value."""
    sel = (E >= E0 - half) & (E <= E0 + half)
    far = np.nanmax(Z[sel][:, x > x.max() - 0.2 * (x.max() - x.min())])
    out = np.full(x.size, np.nan)
    for i in range(x.size):
        col = Z[sel, i]
        if np.isfinite(col).all() and col.max() > h_min * far:
            out[i] = E[sel][np.argmax(col)]
    return out

MAPS = {}
for t in TAGS:
    rec, settings, meas, fine, fit, cpot = series[t]
    E_fb = settings['E_fb_measured']
    h = HYBRID[t]
    x_h = h['x'] / 10.0 + fit['x0']
    top = h['ldos'][-1]
    far = x_h > x_h.max() - 30
    E_far = sw.flat_band_energy(h['energy'], top[:, far].mean(axis=1))
    dE = E_fb - E_far
    loc = np.load(os.path.join(EXAMPLES, f'stacking_wall_tibb_map_{t}.npz'))
    exp = sw.load_line_spectrum(SM4[t]) if SM4[t] and os.path.exists(SM4[t]) else None
    MAPS[t] = dict(x_h=x_h, E_h=h['energy'] + dE, top=top, loc=loc, exp=exp, dE=dE, E_fb=E_fb)
    print(f'[{t}] hybrid flat band far from the wall at {1e3*E_far:+.1f} meV, shifted by {1e3*dE:+.1f} meV to the measured {1e3*E_fb:+.1f} mV')

lo, hi = -0.45, 0.45
for t in TAGS:
    m = MAPS[t]; fit = series[t][4]
    panels = []
    if m['exp'] is not None:
        panels.append(('experiment', m['exp']['dist'], m['exp']['bias'], m['exp']['didv'] * 1e3))
    panels.append(('Poisson solver, local approximation', m['loc']['dist'], m['loc']['bias'], m['loc']['didv']))
    panels.append(('hybrid: k·p wall with the Poisson potentials', m['x_h'], m['E_h'], m['top'] * 1e3))
    fig, axs = plt.subplots(2, len(panels), figsize=(5.2 * len(panels), 7.5), squeeze=False)
    for c, (title, x, E, Z) in enumerate(panels):
        sel = (E >= lo) & (E <= hi)
        xs = (x >= m['loc']['dist'].min()) & (x <= m['loc']['dist'].max())
        Zc = Z[np.ix_(sel, xs)]
        v = np.nanpercentile(Zc, [1, 99.5])
        im = axs[0, c].imshow(Zc, aspect='auto', origin='lower', cmap='plasma', extent=[x[xs].min(), x[xs].max(), lo, hi], vmin=v[0], vmax=v[1])
        fig.colorbar(im, ax=axs[0, c], pad=0.02)
        g = np.gradient(Zc, E[sel], axis=0); vg = np.nanpercentile(np.abs(g), 98)
        im = axs[1, c].imshow(g, aspect='auto', origin='lower', cmap='coolwarm', extent=[x[xs].min(), x[xs].max(), lo, hi], vmin=-vg, vmax=vg)
        fig.colorbar(im, ax=axs[1, c], pad=0.02)
        axs[0, c].set_title(title, fontsize=9)
        for ax in axs[:, c]:
            ax.axvline(fit['x0'], color='w', ls='--', lw=0.8); ax.set_xlabel('distance (nm)'); ax.set_ylabel('bias / energy (V, eV)')
    fig.suptitle(f'line spectrum {t}: dI/dV(V, x) (top) and its bias derivative (bottom)', fontsize=10)
    fig.tight_layout()

## 5. The flat-band peak along the line

Apparent flat-band energy against position for the three descriptions (the peak is followed until it has faded to 15 % of its far value).

In [ ]:
fig, axs = plt.subplots(1, len(TAGS), figsize=(6 * len(TAGS), 3.8), squeeze=False)
for col, t in enumerate(TAGS):
    m = MAPS[t]; fit = series[t][4]; ax = axs[0, col]
    if m['exp'] is not None:
        ax.plot(m['exp']['dist'], 1e3 * peak_track(m['exp']['dist'], m['exp']['bias'], m['exp']['didv'], m['E_fb']), '.', ms=3, color=INK, label='experiment')
    ax.plot(m['loc']['dist'], 1e3 * peak_track(m['loc']['dist'], m['loc']['bias'], m['loc']['didv'], m['E_fb']), '-', color=ORANGE, lw=1.4, label='local approximation (Poisson)')
    ax.plot(m['x_h'], 1e3 * peak_track(m['x_h'], m['E_h'], m['top'], m['E_fb']), '-', color=BLUE, lw=1.4, label='hybrid (k·p wall + Poisson potentials)')
    ax.axvline(fit['x0'], color=MUTED, ls='--', lw=0.8)
    ax.set(xlim=(fit['x0'] - 25, fit['x0'] + 40), xlabel='distance (nm)', ylabel='flat-band peak (mV)', title=f'line spectrum {t}')
    ax.legend(fontsize=7, frameon=False)
fig.tight_layout()

## 6. The tip-induced part (optional, expensive)

The Poisson solver also gives the potential under the apex, U0_l(V) − U_far,l, which moves with the tip. In a line scan it can only renormalise the spectrum under the apex, so putting it into the wall model means one wall calculation per tip position: the bump ΔU_l(x; x_tip) = [U0_l(V_FB, s(x_tip)) − U_far,l(s(x_tip))] exp(−(x − x_tip)²/2w²), with w ≈ √(2 R d) ≈ 5 nm the size of the gated spot, is added to V_l(x) and the top-layer LDOS is read at x_tip. The cell below does this for a few positions when `RUN_TIP_BUMP` is set; at the resolution of section 3 each position costs as much as one hybrid map.

In [ ]:
RUN_TIP_BUMP = False
TIP_POSITIONS_NM = [-15.0, -8.0, -4.0, 0.0, 6.0]        # relative to the wall centre, positive on the (+5) side
W_SPOT_A = 50.0                                          # gated spot, A
if RUN_TIP_BUMP:
    for t in TAGS:
        rec, settings, meas, fine, fit, cpot = series[t]
        E_fb = settings['E_fb_measured']
        L, ell = 4800.0, 10.0 * fit['ell']
        V_of_s = potential_of_s(rec)
        s_vals = np.array(sorted(rec))
        dU = np.array([np.interp(E_fb, rec[s]['bias'], rec[s]['U0'][:, l]) - rec[s]['U_far'][l] for s in s_vals for l in range(6)]).reshape(s_vals.size, 6)
        rows = []
        for xt in TIP_POSITIONS_NM:
            s_t = sw.wall_profile(10.0 * xt, ell, L)
            bump = np.array([np.interp(s_t, s_vals, dU[:, l]) for l in range(6)])
            prof = lambda x, ell=ell: sw.wall_profile(x, ell, L)
            lpp = lambda x, V_of_s=V_of_s, prof=prof, bump=bump, xt=xt: (V_of_s(prof(x)) + bump[:, None] * np.exp(-0.5 * ((np.asarray(x) - 10.0 * xt) / W_SPOT_A) ** 2)[None, :])
            res = sw.ldos_wall(wall_length=ell, length=L, p_cut=0.15, ky_max=0.15, dky=0.0015, sigma=0.005,
                               energy=np.arange(-0.3, 0.3 + 1e-9, 0.0025), x_out=np.array([10.0 * xt]), layer_potential_profile=lpp)
            rows.append((xt, res['energy'], res['ldos'][-1][:, 0]))
            print(f'[{t}] tip at {xt:+.0f} nm: bump of the top layer {1e3*bump[0]:+.1f} meV')
        fig, ax = plt.subplots(figsize=(6, 3.5))
        for i, (xt, E, y) in enumerate(rows):
            ax.plot(E * 1e3, y / rows[-1][2].max() + 0.6 * i, label=f'tip at {xt:+.0f} nm')
        ax.set(xlabel='E − E_F (meV)', ylabel='top-layer LDOS under the tip, offset', title=f'{t}: with the tip-induced bump')
        ax.legend(fontsize=7, frameon=False)
else:
    print('tip-induced bump not run (set RUN_TIP_BUMP = True)')